# Project 3 · Part 1: Project 3 · Part 1: City rents — cleaning listings and building features · Practice

**Open Fraud Labs Academy**

1. Copy your **practice key** from the Practice tab of this lesson on the Academy and paste it into the first code cell.
2. Run the first cell, then complete each exercise and run its check.
3. Every check that passes is saved to your lesson page automatically. Solve all three to unlock the quiz.

Tip: choose **File > Save a copy in Drive** to keep your work.

In [ ]:
# Step 1: paste your practice key between the quotes, then run this cell (Shift + Enter).
# Find your key on the lesson page, in the Practice tab.
OFL_KEY = ""

import pandas as pd, requests
DATA_URL = "https://raw.githubusercontent.com/Odugbile1993/openfraudlab-tiktok/main/data/loans.csv"
COURSE, LESSON = "data-science", 37

def ofl_check(exercise, check_code):
    """Runs the check for one exercise and, if it passes, records it on your Academy lesson page."""
    try:
        exec(check_code, globals())
    except AssertionError as e:
        print("Not quite yet:", e or "compare your result with the task and try again.")
        return
    except NameError as e:
        print("Something the task asks for is missing. Run your code cell first.", e)
        return
    print("Correct! Well done.")
    if not OFL_KEY.strip():
        print("Add your practice key in the first cell and run it, then run this check again to record your progress.")
        return
    try:
        r = requests.post("https://virxrqwxvgsbcnmhdwlp.supabase.co/rest/v1/rpc/practice_report",
                          headers={"apikey": "sb_publishable_8IenXF_4F5ew8hLai8RfVA_dNHq4I72", "Content-Type": "application/json"},
                          json={"p_key": OFL_KEY.strip(), "p_course": COURSE, "p_lesson": LESSON, "p_exercise": exercise}, timeout=20)
        d = r.json()
        if r.ok:
            print(f"Saved to your lesson page: {d['solved']} of {d['total']} exercises solved.")
        else:
            print("Could not save your progress:", d.get("message", r.text))
    except Exception as e:
        print("Could not reach the Academy to save your progress:", e)

print("Ready. Work through the exercises below." if OFL_KEY.strip() else "Paste your practice key above so your progress is saved.")

## Exercise 1: Remove duplicate listings

Count the fully duplicated rows in n_dupes, and store the listings without duplicates in clean.

<details><summary>Hint</summary>

rentals.duplicated() marks repeats; .sum() counts them. rentals.drop_duplicates() removes them.

</details>

In [ ]:
import pandas as pd

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))

n_dupes = ...       # how many fully duplicated rows?
clean = ...         # rentals without the duplicates

print(n_dupes, len(clean))

In [ ]:
# Check exercise 1 (run after your code above)
ofl_check(1, "_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_n = int(_r.duplicated().sum())\nassert n_dupes is not Ellipsis and int(n_dupes) == _n, \"n_dupes should be rentals.duplicated().sum()\"\nassert isinstance(clean, pd.DataFrame), \"clean should be a DataFrame: rentals.drop_duplicates()\"\nassert len(clean) == len(_r) - _n and clean.duplicated().sum() == 0, \"clean should have every duplicated row removed\"\n")

## Exercise 2: Fill missing sizes by type

Fill each missing size_sqm with the median size of that listing's property type, so no sizes are missing.

<details><summary>Hint</summary>

rentals.groupby("property_type")["size_sqm"].transform("median") gives each row its type's median; pass it to .fillna().

</details>

In [ ]:
import pandas as pd

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))
rentals = rentals.drop_duplicates().reset_index(drop=True)

# fill each missing size with the median size of its property type
type_median = ...
rentals["size_sqm"] = ...

print(rentals["size_sqm"].isna().sum())

In [ ]:
# Check exercise 2 (run after your code above)
ofl_check(2, "_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_r = _r.drop_duplicates().reset_index(drop=True)\n_want = _r[\"size_sqm\"].fillna(_r.groupby(\"property_type\")[\"size_sqm\"].transform(\"median\"))\nassert isinstance(rentals[\"size_sqm\"], pd.Series) and rentals[\"size_sqm\"].isna().sum() == 0, \"Every missing size should be filled\"\nassert len(rentals) == len(_r), \"Drop the duplicates first, and keep every other listing\"\nassert ((rentals[\"size_sqm\"].astype(float) - _want).abs() < 1e-9).all(), \"Fill each gap with the median size of that listing's property type (groupby + transform)\"\n")

## Exercise 3: Features and the priciest neighbourhood

Build property_age (the listings are from 2025) and rent_per_sqm, then store the neighbourhood with the highest median rent per square metre in top_neighbourhood.

<details><summary>Hint</summary>

property_age = 2025 - year_built; rent_per_sqm = monthly_rent_usd / size_sqm; then groupby("neighbourhood")["rent_per_sqm"].median().idxmax().

</details>

In [ ]:
import pandas as pd

rentals = pd.read_csv(DATA_URL.replace("loans.csv", "projects/city_rentals.csv"))
rentals = rentals.drop_duplicates().reset_index(drop=True)
rentals["size_sqm"] = rentals["size_sqm"].fillna(rentals.groupby("property_type")["size_sqm"].transform("median"))
rentals["year_built"] = rentals["year_built"].fillna(rentals["year_built"].median())

rentals["property_age"] = ...      # the listings are from 2025
rentals["rent_per_sqm"] = ...      # for exploration only, never a model input
top_neighbourhood = ...                     # neighbourhood with the highest median rent per sqm

print(top_neighbourhood)

In [ ]:
# Check exercise 3 (run after your code above)
ofl_check(3, "_r = pd.read_csv(DATA_URL.replace(\"loans.csv\", \"projects/city_rentals.csv\"))\n_r = _r.drop_duplicates().reset_index(drop=True)\n_r[\"size_sqm\"] = _r[\"size_sqm\"].fillna(_r.groupby(\"property_type\")[\"size_sqm\"].transform(\"median\"))\n_r[\"year_built\"] = _r[\"year_built\"].fillna(_r[\"year_built\"].median())\n_age = 2025 - _r[\"year_built\"]\n_rps = _r[\"monthly_rent_usd\"] / _r[\"size_sqm\"]\nassert isinstance(rentals[\"property_age\"], pd.Series) and ((rentals[\"property_age\"].astype(float) - _age).abs() < 1e-9).all(), \"property_age should be 2025 - year_built\"\nassert isinstance(rentals[\"rent_per_sqm\"], pd.Series) and ((rentals[\"rent_per_sqm\"].astype(float) - _rps).abs() < 1e-6).all(), \"rent_per_sqm should be monthly_rent_usd / size_sqm\"\nassert isinstance(top_neighbourhood, str) and top_neighbourhood == _rps.groupby(_r[\"neighbourhood\"]).median().idxmax(), \"top_neighbourhood should be the neighbourhood with the highest median rent_per_sqm: groupby(\\\"neighbourhood\\\")[...].median().idxmax()\"\n")

---
Done? Go back to the lesson on [openfraudlabs.com/academy](https://openfraudlabs.com/academy/lesson/?course=data-science&n=37) and take the quiz.